In [0]:
# ============================================
# NOTEBOOK 2: Cleaning & Transformation
# ============================================

from pyspark.sql.functions import col, year, month, quarter, when, count, avg
from pyspark.sql.functions import round as spark_round

print("="*70)
print("TRANSFORMACIÓN")
print("="*70)


In [0]:
# ============================================
# Cargar Bronze
# ============================================

df = spark.table("bronze_property_sales")

print(f" Registros: {df.count():,}")

In [0]:
# ============================================
# LIMPIEZA: Eliminar precios inválidos
# ============================================

print("\n" + "="*70)
print("LIMPIEZA DE PRECIOS")
print("="*70)

print(f"Registros antes: {df.count():,}")

# Eliminar precios nulos, negativos o cero
df_clean = df.filter(
    (col("price").isNotNull()) & 
    (col("price") > 0)
)

registros_eliminados = df.count() - df_clean.count()
print(f"Registros después: {df_clean.count():,}")
print(f"Registros eliminados: {registros_eliminados:,}")

# Ver estadísticas de precios limpios
print("\n--- Estadísticas de precios limpios ---")
df_clean.select("price").describe().show()

In [0]:
# ============================================
# TRANSFORMACIÓN: Extraer componentes de fecha
# ============================================

print("\n" + "="*70)
print("EXTRACCIÓN DE COMPONENTES TEMPORALES")
print("="*70)

# Ya tenemos date_of_transfer como timestamp, solo extraemos componentes
df_clean = df_clean.withColumn("year", year("date_of_transfer")) \
                   .withColumn("month", month("date_of_transfer")) \
                   .withColumn("quarter", quarter("date_of_transfer"))

print("\n✓ Columnas temporales creadas: year, month, quarter")

# Verificar
df_clean.select("date_of_transfer", "year", "month", "quarter").show(10)

# Ver distribución por año
print("\n--- Transacciones por año ---")
df_clean.groupBy("year").count().orderBy("year").show()

In [0]:
# ============================================
# TRANSFORMACIÓN: Categorías de precio
# ============================================

print("\n" + "="*70)
print("CATEGORIZACIÓN DE PRECIOS")
print("="*70)

# Categorías basadas en el mercado UK
df_clean = df_clean.withColumn(
    "price_category",
    when(col("price") < 200000, "Low")
    .when((col("price") >= 200000) & (col("price") < 500000), "Medium")
    .when((col("price") >= 500000) & (col("price") < 1000000), "High")
    .otherwise("Premium")
)

print("\n--- Distribución por categoría de precio ---")
df_clean.groupBy("price_category") \
        .agg({"price": "count", "price": "avg"}) \
        .withColumnRenamed("count(price)", "total_transacciones") \
        .withColumnRenamed("avg(price)", "precio_promedio") \
        .orderBy("precio_promedio") \
        .show()

In [0]:
# ============================================
# TRANSFORMACIÓN: Decodificar códigos
# ============================================

print("\n" + "="*70)
print("DECODIFICACIÓN DE CÓDIGOS")
print("="*70)

# Tipo de propiedad
df_clean = df_clean.withColumn(
    "property_type_desc",
    when(col("property_type") == "D", "Detached")
    .when(col("property_type") == "S", "Semi-Detached")
    .when(col("property_type") == "T", "Terraced")
    .when(col("property_type") == "F", "Flat/Apartment")
    .when(col("property_type") == "O", "Other")
    .otherwise("Unknown")
)

# Nueva construcción vs Reventa
df_clean = df_clean.withColumn(
    "property_age",
    when(col("old_new") == "Y", "New Build")
    .otherwise("Resale")
)

# Tipo de tenencia
df_clean = df_clean.withColumn(
    "tenure_type",
    when(col("duration") == "F", "Freehold")
    .when(col("duration") == "L", "Leasehold")
    .otherwise("Unknown")
)

print("\n✓ Códigos decodificados")

# Verificar
print("\n--- Distribución por tipo de propiedad ---")
df_clean.groupBy("property_type_desc").count().orderBy("count", ascending=False).show()

print("\n--- Nueva construcción vs Reventa ---")
df_clean.groupBy("property_age").count().show()

print("\n--- Freehold vs Leasehold ---")
df_clean.groupBy("tenure_type").count().show()

In [0]:
# ============================================
# LIMPIEZA: Valores nulos en ciudades
# ============================================

print("\n" + "="*70)
print("LIMPIEZA DE CIUDADES")
print("="*70)

# Contar nulos antes
nulos_antes = df_clean.filter(col("town_city").isNull()).count()
print(f"Ciudades nulas antes: {nulos_antes:,}")

# Rellenar con 'Unknown'
df_clean = df_clean.withColumn(
    "town_city",
    when(col("town_city").isNull(), "Unknown").otherwise(col("town_city"))
)

# Verificar
nulos_despues = df_clean.filter(col("town_city") == "Unknown").count()
print(f"Ciudades 'Unknown' después: {nulos_despues:,}")

In [0]:
# ============================================
# GUARDAR TABLA SILVER
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA SILVER")
print("="*70)

spark.sql("DROP TABLE IF EXISTS silver_property_sales")

# Seleccionar columnas finales
df_silver = df_clean.select(
    "transaction_id",
    "price",
    "date_of_transfer",
    "year",
    "month",
    "quarter",
    "price_category",
    "postcode",
    "property_type",
    "property_type_desc",
    "property_age",
    "tenure_type",
    "town_city",
    "district",
    "county"
)

# Guardar como tabla Silver

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_property_sales")


print("\n✅ Tabla Silver creada: silver_property_sales")
print(f"✓ Total de registros: {df_silver.count():,}")

# Ver muestra
print("\nPrimeras 5 filas de Silver:")
display(df_silver.limit(5))

In [0]:
# ============================================
# TABLA GOLD 1: Análisis por ciudad
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - CIUDADES")
print("="*70)

df_gold_city = spark.sql("""
SELECT 
    town_city,
    COUNT(*) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(MIN(price), 0) as min_price,
    ROUND(MAX(price), 0) as max_price,
    PERCENTILE_APPROX(price, 0.5) as median_price
FROM silver_property_sales
WHERE town_city != 'Unknown'
GROUP BY town_city
HAVING COUNT(*) >= 10
ORDER BY avg_price DESC
""")

# Guardar con overwriteSchema
df_gold_city.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_city_prices")

print("\n✅ Tabla Gold creada: gold_city_prices")
print(f"✓ Total de ciudades: {df_gold_city.count():,}")
print("\n--- Top 10 ciudades más caras ---")
df_gold_city.show(10, truncate=False)

In [0]:
# ============================================
# TABLA GOLD 2: Evolución temporal
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - EVOLUCIÓN TEMPORAL")
print("="*70)

df_gold_temporal = spark.sql("""
SELECT 
    year,
    month,
    COUNT(*) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(STDDEV(price), 0) as std_price
FROM silver_property_sales
GROUP BY year, month
ORDER BY year, month
""")

# Guardar
df_gold_temporal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_temporal_trends")

print("\n✅ Tabla Gold creada: gold_temporal_trends")
print("\nÚltimos 12 meses:")
df_gold_temporal.orderBy("year", "month", ascending=False).show(12)

In [0]:
# ============================================
# TABLA GOLD 3: Análisis por tipo de propiedad
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - TIPO DE PROPIEDAD")
print("="*70)

df_gold_property = spark.sql("""
SELECT 
    property_type_desc,
    property_age,
    tenure_type,
    COUNT(*) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(MIN(price), 0) as min_price,
    ROUND(MAX(price), 0) as max_price
FROM silver_property_sales
GROUP BY property_type_desc, property_age, tenure_type
ORDER BY avg_price DESC
""")

# Guardar
df_gold_property.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_property_analysis")

print("\n✅ Tabla Gold creada: gold_property_analysis")
df_gold_property.show(20, truncate=False)

In [0]:
# ============================================
# RESUMEN FINAL DEL NOTEBOOK 2
# ============================================

print("\n" + "="*70)
print("RESUMEN FINAL - ARQUITECTURA MEDALLION")
print("="*70)

print("\nTablas creadas:")
spark.sql("SHOW TABLES").filter("tableName LIKE '%property%' OR tableName LIKE '%gold%'").show(truncate=False)

print("\nEstadísticas:")
print(f"  Bronze (raw): {spark.table('workspace.default.price_paid_data').count():,} registros")
print(f"  Silver (clean): {spark.table('silver_property_sales').count():,} registros")
print(f"  Gold - Ciudades: {spark.table('gold_city_prices').count():,} ciudades")
print(f"  Gold - Temporal: {spark.table('gold_temporal_trends').count():,} periodos")
print(f"  Gold - Propiedades: {spark.table('gold_property_analysis').count():,} combinaciones")

print("\n✅ Notebook 2 completado exitosamente")